# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [22]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [6]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [8]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {'table': # ENTER A TABLE COLUMNS HERE,
        'definition': # ENTER A TABLE DEFINITATIONS HERE
df = pd.DataFrame(data)
print(df)

       table                                         definition
0  employees                      Employee information, name...
1     salary                       Salary details for each year
2    studies  Educational studies, name of the institution, ...


In [10]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [14]:
print(text_tables)

employees: Employee information, name...
salary: Salary details for each year
studies: Educational studies, name of the institution, type of studies, level


In [16]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [18]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question=#ENTER YOUR QUERY HERE)

In [24]:
print(return_OAI(pqt1))

```json
{
    "tables": ["employees", "salary"]
}
```


In [26]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question=#ENTER YOUR QUERY HERE)

In [28]:
print(return_OAI(pqt3))

```json
{
    "tables": {
        "employees": "Employee information",
        "salary": "Salary details for each year",
        "studies": "Educational studies"
    }
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [2]:

import os
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# Load API key
_ = load_dotenv(find_dotenv())
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Create OpenAI client
client = OpenAI(api_key=OPENAI_API_KEY)

# Function to call the model
def return_OAI(user_message):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {
                "role": "system",
                "content": "You are an expert SQL assistant. Your job is to identify which database tables are needed to answer a user's question."
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content

# 1. Define thedatabase tables and their meanings
data = {
    "table": [
        "customers",
        "orders",
        "products",
        "order_items",
        "suppliers",
        "categories",
        "employees",
        "payments",
        "shipping",
    ], 
    "definition": [
        "Contains customer information such as customer_id, customer_name, email, country and city.",
        "Contains order information such as order_id, customer_id, employee_id, order_date and order_status.",
        "Contains details of each order line, including order_id, product_id, quantity, unit_price and discount.",
        "Contains product information such as product_id, product_name, category_id, supplier_id and price.",
        "Contains product category information such as category_id and category_name.",
        "Contains supplier information such as supplier_id, supplier_name, contact_name and country.",
        "Contains employee information such as employee_id, employee_name, department and hire_date.",
        "Contains payment information such as payment_id, order_id, payment_date, payment_method and amount.",
        "Contains shipping information such as shipping_id, order_id, shipping_date, delivery_date and shipping_company."
    ]
}

df = pd.DataFrame(data)

# 2. Convert table definitions to text for the prompt
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

# 3. Define the prompt template
prompt_question_tables = """
Given the following database tables and their content definitions:

### Tables
{tables}

Tell me which tables would be necessary to query with SQL to answer the user's question.

Rules:
- Return only the required tables.
- Do not invent table names.
- Do not include unnecessary tables.
- Return the answer in valid JSON format only.

Use this format:
{{
  "tables": ["table_1", "table_2"]
}}

### User Question:
{question}
"""

# 4. Final Questions
questions = [
    "Which customers placed orders in 2023?",
    "What are the total sales by product category?",
    "List the employees who have processed orders for customers in the USA.",
    "How many products are supplied by each supplier?",
    "How much revenue did each employee generate?"
]

# 5. Run the model for each question
for question in questions:
    prompt = prompt_question_tables.format(tables=text_tables, question=question)
    print(f"Question: {question}")
    print("Model Answer:", return_OAI(prompt))
    print("\n")

Question: Which customers placed orders in 2023?
Model Answer: {
  "tables": ["customers", "orders"]
}


Question: What are the total sales by product category?
Model Answer: {
  "tables": ["products", "order_items"]
}


Question: List the employees who have processed orders for customers in the USA.
Model Answer: {
  "tables": ["customers", "orders", "employees"]
}


Question: How many products are supplied by each supplier?
Model Answer: {
  "tables": ["order_items"]
}


Question: How much revenue did each employee generate?
Model Answer: {
  "tables": ["orders", "products"]
}




Report — SQL Query from Table Names

This lab shows how a language model can identify the tables needed to answer a user’s question with SQL.

The model receives a list of table names and short descriptions. Then, based on the question, it selects only the relevant tables.

For example, if the question is about customers who placed orders, the model should choose customers and orders.

The main idea is that selecting the right tables before writing the SQL query makes the process easier and reduces mistakes.

The lab also shows that the prompt must be clear. If the instructions are vague, the model may select unnecessary tables or return the answer in the wrong format.

In conclusion, this lab is useful because it shows an important first step in Text-to-SQL systems: choosing the correct tables before generating the SQL query.